In [2]:
import torch


In [3]:
import math
import torch.nn as nn


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


In [5]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "tiny_shakespeare.txt"
)
print("Downloaded!")


Downloaded!


In [6]:
text = ""
# with open("file.txt", 'r', encoding='utf-8') as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         content = line.split(sep = '-', maxsplit=1)
#         if '<Media omitted>' in content:
#             continue
#         if len(content)<2:
#             text += "\n" + line
#             continue
#         text += "\n"
#         text += content[1].strip()

with open("tiny_shakespeare.txt", 'r', encoding='utf-8') as f:
    for line in f:
        text += line

chars = list(set(text))
chars.sort()
stoi = {char : ind for ind, char in enumerate(chars)}
itos = {ind : char for ind, char in enumerate(chars)}
vocab_size = len(stoi)

def encode(text):
    return [stoi[char] for char in text]

def decode(embedding):
    return [itos[ind] for ind in embedding]

def train_test_split(data, train=0.9):
    n = int(train*len(data))
    tr = data[:n]
    val = data[n:len(data)]
    return tr, val


In [7]:
print(text[:1000])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [28]:
data = torch.tensor(encode(text), dtype=torch.long)
train_data, val_data = train_test_split(data)

torch.manual_seed(42)
batch_size = 64
block_size = 256

def get_batch(split):
    data = train_data if split=='train' else val_data
    inds = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in inds]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in inds]).to(device)
    return x, y

xb, yb = get_batch('train')
print(xb)
print(yb)


tensor([[54, 43, 63,  ..., 54, 43, 63],
        [57,  1, 61,  ..., 47, 52,  1],
        [57,  1, 58,  ..., 46, 39, 58],
        ...,
        [23, 17,  1,  ..., 24, 17, 10],
        [43, 50, 54,  ...,  1, 21, 51],
        [63, 53, 59,  ..., 63,  1, 58]], device='cuda:0')
tensor([[43, 63,  1,  ..., 43, 63, 12],
        [ 1, 61, 53,  ..., 52,  1, 61],
        [ 1, 58, 46,  ..., 39, 58, 46],
        ...,
        [17,  1, 27,  ..., 17, 10,  0],
        [50, 54,  1,  ..., 21, 51, 54],
        [53, 59,  1,  ...,  1, 58, 39]], device='cuda:0')


In [29]:
# for i in range(batch_size):
#     for j in range(block_size):
#         context = xb[i, :j+1]
#         target = yb[i, j]


In [30]:
class AdamOptimizer:
    def __init__(self, params, lr=3e-4, eps=1e-8, betas=(0.9, 0.999)):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.beta1 = betas[0]
        self.beta2 = betas[1]
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                self.m[i] = self.beta1*self.m[i] + (1-self.beta1)*p.grad
                m_hat = self.m[i]/(1-self.beta1**self.t)
                self.v[i] = self.beta2*self.v[i] + (1-self.beta2)*(p.grad**2)
                v_hat = self.v[i]/(1-self.beta2**self.t)
                p -= (m_hat*self.lr)/(v_hat.sqrt()+self.eps)
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [31]:
def softmax(x):
    exp_x = torch.exp(x)
    return exp_x/exp_x.sum(dim=-1, keepdim=True) # why -1? -> the matrix is batchxqueryxkey, so we have to softmax over all keys

def causal_softmax(x): # matrix = (batch, block_size, block_size) - attention matrix
    mask = torch.tril(torch.ones(block_size, block_size)).to(device)
    x_mask = x.masked_fill(mask == 0, -float('inf'))
    return softmax(x_mask)  


In [40]:
n_embd = 384

class Attention(nn.Module):
    def __init__(self, heads=6):
        super().__init__()
        self.heads = heads
        self.head_size = n_embd//self.heads
        self.W_q = nn.Linear(n_embd, n_embd, bias=False)
        self.W_k = nn.Linear(n_embd, n_embd, bias=False)
        self.W_v = nn.Linear(n_embd, n_embd, bias=False)
        self.W_o = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))) # register_buffer - tells pytorch that it is not trainable
    
    def multi_head(self, mat):
        B = mat.shape[0] # batch_size
        mat = mat.view(B, block_size, self.heads, self.head_size)
        return mat.transpose(1, 2)  # (B, heads, T, head_size)
    
    def forward(self, batch): # batch = (batch_size, block_size, embedding_size), W_q = (embedding_size, output_size=n_embd)
        B = batch.shape[0]
        Q = self.W_q(batch) # (batch_size, block_size, n_embd)
        K = self.W_k(batch)
        V = self.W_v(batch)
        Q = self.multi_head(Q)
        K = self.multi_head(K)
        V = self.multi_head(V) 
        w = Q@K.transpose(-2, -1) # we only transpose within a head within a block - (B, heads, T, T)
        w = w/math.sqrt(self.head_size) 
        # W = causal_softmax(w2) # (batch_size, heads, block_size, block_size)
        w = w.masked_fill(self.mask == 0, -float('inf'))
        W = torch.softmax(w, dim=3) # w - (batch_size, heads, block_size, block_size) - we softmax across fourth dim
        new_tokens = W @ V # (B, heads, T, head_size)
        new_tokens = new_tokens.transpose(1, 2) # (B, T, heads, head_size)
        new_tokens = new_tokens.reshape(B, block_size, -1)
        new_tokens = self.W_o(new_tokens)
        return new_tokens


In [41]:
def relu(x): return torch.clamp(x, min=0)
def gelu(x): return x*0.5*(1+torch.erf(x/math.sqrt(2)))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 =  nn.Linear(n_embd, 4*n_embd, bias=True)
        self.l2 = nn.Linear(4*n_embd, n_embd, bias=True)

    def forward(self, batch):
        x1 = self.l1(batch)
        x2 = gelu(x1)
        x3 = self.l2(x2)
        return x3


In [42]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = Attention()
        self.attention_norm = nn.LayerNorm(n_embd)
        self.ff = FeedForward()
        self.ff_norm = nn.LayerNorm(n_embd)
        self.dropout = nn.Dropout(0.2) # zeroes out some of the activations (tensor values)
    
    def forward(self, batch):
        batch = batch + self.dropout(self.attention(self.attention_norm(batch))) # nn.Module runs self.forward automatically
        batch = batch + self.dropout(self.ff(self.ff_norm(batch)))
        return batch
    

In [43]:
def loss(logits, y):
    loss_fn = nn.CrossEntropyLoss()
    return loss_fn(logits.view(-1, vocab_size), y.view(-1))


In [44]:
class Model(nn.Module):
    def __init__(self, layers=6):
        super().__init__()
        self.n_layers = layers
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block() for _ in range(self.n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=True)
    
    def forward(self, batch): # batch = [[8 numbers], [], [], []] - 4 batches
        tokens = self.token_embedding(batch) # (batch_size, block_size, n_embd)
        tokens = tokens + self.position_embedding(torch.arange(block_size).to(device)) # (batch, block, n_embd) + (block, n_embd)
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.ln_f(tokens)
        logits = self.lm_head(tokens)
        return logits
    



In [45]:
def train_step(model : Model, optimizer : AdamOptimizer):
    model.train()
    optimizer.zero_grad()
    xb, yb = get_batch('train')
    logits = model(xb)
    step_loss = loss(logits, yb)
    step_loss.backward()
    optimizer.step()
    return step_loss.item()

def val_step(model):
    model.eval()
    with torch.no_grad():
        xb, yb = get_batch('val')
        logits = model(xb)
        step_loss = loss(logits, yb)
    return step_loss.item()

def accuracy(model, samples=1000):
    ...

def epoch(model, optimizer):
    val_interval = 100
    steps = len(train_data)//(batch_size*block_size)
    net_tr_loss = 0
    net_val_loss = 0
    for i in range(1, steps+1):
        tr_loss = train_step(model, optimizer)
        net_tr_loss += tr_loss
        if i%val_interval==0:
            val_loss = val_step(model)
            net_val_loss += val_loss
    return net_tr_loss/steps, net_val_loss/(steps//val_interval)

def train(epochs=1):
    model = torch.compile(Model().to(device))
    # optimizer = AdamOptimizer(model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    for n_epoch in range(1, epochs+1):
        tr_loss, val_loss = epoch(model, optimizer)
        print(f"Epoch {n_epoch} | Train Loss: {tr_loss} | Val Loss: {val_loss}")
    return model

def train_iters(max_iters=3400, val_interval=200):
    model = torch.compile(Model().to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    for n_iter in range(1, max_iters+1):
        tr_loss = train_step(model, optimizer)
        if n_iter%val_interval==0:
            val_loss = val_step(model)
            print(f"Step {n_iter:5d}/{max_iters} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f}")
    return model

model = train_iters()


Step   200/3400 | Train Loss: 2.3765 | Val Loss: 2.3975
Step   400/3400 | Train Loss: 2.0074 | Val Loss: 2.0874
Step   600/3400 | Train Loss: 1.7639 | Val Loss: 1.8578
Step   800/3400 | Train Loss: 1.5949 | Val Loss: 1.7416
Step  1000/3400 | Train Loss: 1.5158 | Val Loss: 1.6938
Step  1200/3400 | Train Loss: 1.4458 | Val Loss: 1.6783
Step  1400/3400 | Train Loss: 1.4259 | Val Loss: 1.6082
Step  1600/3400 | Train Loss: 1.3973 | Val Loss: 1.5951
Step  1800/3400 | Train Loss: 1.3090 | Val Loss: 1.5999
Step  2000/3400 | Train Loss: 1.2884 | Val Loss: 1.5367
Step  2200/3400 | Train Loss: 1.2728 | Val Loss: 1.4898
Step  2400/3400 | Train Loss: 1.2558 | Val Loss: 1.5201
Step  2600/3400 | Train Loss: 1.2040 | Val Loss: 1.4965
Step  2800/3400 | Train Loss: 1.2003 | Val Loss: 1.4990
Step  3000/3400 | Train Loss: 1.1910 | Val Loss: 1.5107
Step  3200/3400 | Train Loss: 1.1795 | Val Loss: 1.4739
Step  3400/3400 | Train Loss: 1.1408 | Val Loss: 1.5683


In [50]:
def generate(model, max_new_tokens=1000, temperature=1):
    model.eval()
    idx = torch.zeros((1, 1), dtype=torch.long).to(device)
    for _ in range(max_new_tokens):
        if idx.shape[1] < block_size:
            pad = torch.zeros(1, block_size - idx.shape[1], dtype=torch.long).to(device)
            idx_cond = torch.cat([pad, idx], dim=1)
        else:
            idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        logits = logits[:, -1, :] / temperature  # last token, apply temperature
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return ''.join(decode(idx[0].tolist()))

print(generate(model))



STRKINGHAM:
There has your pretty.

FRIAR LAUMENCE:
Too, done love him.

DUKE VINCENTIO:
But is this too has grown your mouthsuad.

STANLEY:
Shekes, her hear the shew.

GREGORY:
Well, must her day; we doubt the cast
tire some in virtue friends.

DUKE VINCENTIO:
Sir, I think, I can afford; for he
should be happy to with him. Crowns he wakes.

LUCIO:

Clown:
But where is a vise man could that have present size
must another pay the babe poison. Pray you, sir;
there were lay 'twas a knee--shall; but, which, if
we have no human but words, a three gentleman to sin
far errays.

Clown:
Very indeed I fear thee, you would speak so look upon you, if
it were he is, and in the rebel in your crown; you would say,
an it a parpetual who thrust did stand the lunh, but
but fearing some sound many yieldings as
you; she have to your sroock's kindness to me and
be but cry.

Shepherd:
You must not prove it.

First Servant:
If it must be prisoner to come by the gentle queen.

Shepherd:
I would thou dost lik